https://github.com/neural-maze/agentic-patterns-course/blob/main/notebooks/tool_pattern.ipynb

In [11]:
import os
import json
import re

from pprint import pprint
from ollama import Client
from IPython.display import display_markdown

In [4]:
MODEL = "llama3.2:3b"
CLIENT = Client()

# Define the System Prompt as a constant
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags. 
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug 
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>}
</tool_call>

Here are the available tools:

<tools> {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location location (str): The city and state, e.g. Madrid, Barcelona unit (str): The unit. It can take two values; 'celsius', 'fahrenheit'",
    "parameters": {
        "properties": {
            "location": {
                "type": "str"
            },
            "unit": {
                "type": "str"
            }
        }
    }
}
</tools>
"""

In [6]:
def get_current_weather(location: str, unit: str):
	"""
	Get the current weather in a given location

	location (str): The city and state, e.g. Madrid, Barcelona
	unit (str): The unit. It can take two values; "celsius", "fahrenheit"
	"""
	if location == "Madrid":
		return json.dumps({"temperature": 25, "unit": unit})

	else:
		return json.dumps({"temperature": 58, "unit": unit})

get_current_weather(location="Madrid", unit="celsius")

'{"temperature": 25, "unit": "celsius"}'

In [7]:
tool_chat_history = [
    {
        "role": "system",
        "content": TOOL_SYSTEM_PROMPT
    }
]
agent_chat_history = []

user_msg = {
    "role": "user",
    "content": "What's the current temperature in Madrid, in Celsius?"
}

tool_chat_history.append(user_msg)
agent_chat_history.append(user_msg)

output = CLIENT.chat(
    messages=tool_chat_history,
    model=MODEL
).message.content

print(output)

<tool_call>
{"name": "get_current_weather","arguments": {"location": "Madrid", "unit": "celsius"}}
</tool_call>


In [12]:
# Parse the LLM output
# Load the output as a proper Python dict

def parse_tool_call_str(tool_call_str: str):
    pattern = r'</?tool_call>'
    clean_tags = re.sub(pattern, '', tool_call_str)
    
    try:
        tool_call_json = json.loads(clean_tags)
        return tool_call_json
    except json.JSONDecodeError:
        return clean_tags
    except Exception as e:
        print(f"Unexpected error: {e}")
        return "There was some error parsing the Tool's output"



parsed_output = parse_tool_call_str(output)
parsed_output


{'name': 'get_current_weather',
 'arguments': {'location': 'Madrid', 'unit': 'celsius'}}

In [14]:
result = get_current_weather(**parsed_output["arguments"])
result

'{"temperature": 25, "unit": "celsius"}'

In [17]:
agent_chat_history.append({
    "role": "user",
    "content": f"Observation: {result}"
})

output_msg = CLIENT.chat(
    messages=agent_chat_history,
    model=MODEL
).message.content

pprint(output_msg)

'The current temperature in Madrid is 25°C.'


In [21]:
get_current_weather.__name__, get_current_weather.__doc__

('get_current_weather',
 '\n\tGet the current weather in a given location\n\n\tlocation (str): The city and state, e.g. Madrid, Barcelona\n\tunit (str): The unit. It can take two values; "celsius", "fahrenheit"\n\t')

In [24]:
for k, v in get_current_weather.__annotations__.items():
    print(k, v, v.__name__)

location <class 'str'> str
unit <class 'str'> str
